In [1]:
# relative path
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(config_path="../configs"):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg))

/tmp/ipykernel_1399524/941803802.py:5: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="../configs"):


paths:
  nako_dir: /nfs/data/nii/data0/GNC/GNC_759
  nnUNet_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/



In [2]:
# create nnUNet_raw, nnUNet_preprocessed, nnUNet_results
import os
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_raw", exist_ok=True)
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_preprocessed", exist_ok=True)
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_results", exist_ok=True)

raw_ds_dir = os.path.join(cfg.paths.nnUNet_dir, "nnUNet_raw/Dataset001_Nako")
os.makedirs(raw_ds_dir, exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "imagesTr"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "labelsTr"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "imagesTs"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "labelsTs"), exist_ok=True)

In [3]:
nb_subjects = 100
from pathlib import Path
import glob
img_base_path = Path(cfg.paths.nako_dir) / "links"
mask_base_path = Path(cfg.paths.nako_dir) / "data"
img_target = "30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.nii"
mask_target = "30/wholebody/total_vibe_and_ours_seg.nii"
subjects = [e.name for e in os.scandir(img_base_path) if e.is_dir()][:50000]
# keep subdirs where wholebody/total_vibe_and_ours_seg.nii.gz" exists
valid_subjects = []
for subject in subjects:
    mask_path = os.path.join(mask_base_path, subject, mask_target)
    if os.path.exists(mask_path):
        valid_subjects.append(subject)
print("Valid subjects:", len(valid_subjects))
subjects = valid_subjects[:nb_subjects]
subjects_train = subjects[:int(0.8*len(subjects))]
subjects_test = subjects[int(0.8*len(subjects)):]
print("Train subjects:", len(subjects_train))
print("Test subjects:", len(subjects_test))

Valid subjects: 966
Train subjects: 80
Test subjects: 20


In [4]:
import nibabel as nib
import numpy as np
uniques_values = []
for subject in subjects:
    mask_path = os.path.join(mask_base_path, subject, mask_target)
    mask = nib.load(mask_path)
    mask_data = mask.get_fdata()
    unique = np.unique(mask_data)
    uniques_values.append(unique)


In [8]:
# distribution of unique values in the masks
lens=[len(unique) for unique in uniques_values]
print("Unique values distribution:", np.bincount(lens))


Unique values distribution: [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  1  0 36 60  3]


In [9]:
import nibabel as nib
import numpy as np

def process_subject(subject, img_base_path, mask_base_path, img_target, mask_target, images_dir, labels_dir):
    img_path = img_base_path / subject / img_target
    mask_path = mask_base_path / subject / mask_target

    # --- Images: split 4D -> 4x 3D channel files ---
    print(f"Looking for images in: {img_path}")
    files = glob.glob(str(img_path))
    print(f"Subject: {subject}, Files found: {len(files)}")

    if len(files) > 0:
        img = nib.load(files[0])
        data = img.get_fdata()  # (320, 260, 316, 4)

        if data.ndim == 4:
            for ch in range(data.shape[-1]):
                channel_data = data[..., ch].astype(np.float32)
                new_img = nib.Nifti1Image(channel_data, img.affine, img.header)
                new_img.header.set_data_shape(channel_data.shape)
                dst = os.path.join(images_dir, f"{subject}_{ch:04d}.nii.gz")
                nib.save(new_img, dst)
                print(f"  Saved channel {ch} -> {dst}")
        else:
            # Already 3D (single channel), just copy as _0000
            dst = os.path.join(images_dir, f"{subject}_0000.nii.gz")
            shutil.copy(files[0], dst)
            print(f"  Copied single-channel image -> {dst}")

    # --- Mask: enforce 3D header and save ---
    print(f"Looking for masks in: {mask_path}")
    mask_files = glob.glob(str(mask_path))
    print(f"Subject: {subject}, Mask files found: {len(mask_files)}")

    if len(mask_files) > 0:
        mask = nib.load(mask_files[0])
        mask_data = mask.get_fdata().astype(np.uint8)

        # Squeeze out any phantom 4th dim
        if mask_data.ndim == 4:
            mask_data = mask_data[..., 0]

        new_mask = nib.Nifti1Image(mask_data, mask.affine, mask.header)
        new_mask.header.set_data_shape(mask_data.shape)
        dst = os.path.join(labels_dir, f"{subject}.nii.gz")
        nib.save(new_mask, dst)
        print(f"  Saved mask -> {dst}")


for subject in subjects_train:
    process_subject(
        subject,
        img_base_path, mask_base_path,
        img_target, mask_target,
        images_dir=os.path.join(raw_ds_dir, "imagesTr"),
        labels_dir=os.path.join(raw_ds_dir, "labelsTr"),
    )

for subject in subjects_test:
    process_subject(
        subject,
        img_base_path, mask_base_path,
        img_target, mask_target,
        images_dir=os.path.join(raw_ds_dir, "imagesTs"),
        labels_dir=os.path.join(raw_ds_dir, "labelsTs"),
    )

Looking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100121/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.nii
Subject: 100121, Files found: 1
  Saved channel 0 -> /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/nnUNet_raw/Dataset001_Nako/imagesTr/100121_0000.nii.gz
  Saved channel 1 -> /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/nnUNet_raw/Dataset001_Nako/imagesTr/100121_0001.nii.gz
  Saved channel 2 -> /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/nnUNet_raw/Dataset001_Nako/imagesTr/100121_0002.nii.gz
  Saved channel 3 -> /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/nnUNet_raw/Dataset001_Nako/imagesTr/100121_0003.nii.gz
Looking for masks in: /nfs/data/nii/data0/GNC/GNC_759/data/100121/30/wholebody/total_vibe_and_ours_seg.nii
Subject: 100121, Mask files found: 1
  Saved mask -> /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/n

export nnUNet_raw="/nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/nnUNet_raw"
export nnUNet_preprocessed="/nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/nnUNet_preprocessed"
export nnUNet_results="/nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/nnUNet_results"

uv run nnUNetv2_plan_and_preprocess -d 2 --verify_dataset_integrity -pl nnUNetPlannerResEncM -gpu_memory_target 40
uv run nnUNetv2_train 2 3d_fullres 0